# label

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import cohen_kappa_score

## data frame

In [ ]:
origindf = pd.read_excel('expert_label.xlsx')

In [ ]:
LABEL_COLS = [
    "expert_1_label",
    "expert_2_label",
    "expert_3_label",
    "final_label"
]
MIDPOINT_SCORE = {
    1: 0.125,
    2: 0.375,
    3: 0.625,
    4: 0.875,
}

In [ ]:
def plot_expert_distribution(
    df: pd.DataFrame,
    save_path=None,
    figsize=(4.5, 3.5),
):

    experts = {
        "Expert 1": "expert_1_label",
        "Expert 2": "expert_2_label",
        "Expert 3": "expert_3_label",
    }

    # 专家标签现在直接使用1–4
    labels = [1, 2, 3, 4]

    counts = pd.DataFrame(
        {
            expert: (
                df[column]
                .value_counts()
                .reindex(
                    labels,
                    fill_value=0,
                )
            )
            for expert, column in experts.items()
        }
    )

    counts.index = [
        f"Label {label}"
        for label in labels
    ]

    fig, ax = plt.subplots(
        figsize=figsize
    )

    x = np.arange(len(experts))
    width = 0.22

    for i, label in enumerate(counts.index):

        values = counts.loc[label].values

        bars = ax.bar(
            x + (i - 1.5) * width,
            values,
            width,
            label=str(labels[i]),
        )

        ax.bar_label(
            bars,
            padding=2,
        )

    ax.set_xticks(x)
    ax.set_xticklabels(
        experts.keys()
    )

    ax.set_ylabel("Count")

    ax.legend(
        title="Relevance level",
        ncol=4,
        loc="upper center",
        bbox_to_anchor=(0.5, 1.32),
        frameon=True,
    )

    ax.set_ylim(
        0,
        counts.values.max() * 1.15,
    )

    plt.tight_layout(
        rect=[0, 0, 1, 0.90]
    )

    if save_path:
        plt.savefig(
            save_path,
            dpi=600,
            bbox_inches="tight",
        )

    plt.show()

    return counts
plot_expert_distribution(origindf)

In [ ]:
def plot_qwk(
    df: pd.DataFrame,
    figsize=(3.5, 3.5),
):
    cols = [
        "expert_1_label",
        "expert_2_label",
        "expert_3_label",
    ]

    names = [
        "Expert 1",
        "Expert 2",
        "Expert 3",
    ]

    qwk = np.eye(3)

    for i in range(3):
        for j in range(i + 1, 3):

            score = cohen_kappa_score(
                df[cols[i]],
                df[cols[j]],
                weights="quadratic",
            )

            qwk[i, j] = score
            qwk[j, i] = score

    fig, ax = plt.subplots(figsize=figsize)

    im = ax.imshow(
        qwk,
        cmap="viridis",
        vmin=0,
        vmax=1,
    )

    ax.set_xticks(range(3))
    ax.set_yticks(range(3))

    ax.set_xticklabels(names, rotation=30, ha="right")
    ax.set_yticklabels(names)

    for i in range(3):
        for j in range(3):
            ax.text(
                j,
                i,
                f"{qwk[i,j]:.2f}",
                ha="center",
                va="center",
                fontsize=10,
                color="black" if qwk[i,j] > 0.65 else "white",
            )

    cbar = plt.colorbar(
        im,
        ax=ax,
        fraction=0.05,
        pad=0.04,
    )

    cbar.set_label("QWK")

    plt.tight_layout()
    plt.show()
plot_qwk(origindf)